# Co-Training with Qwen3-VL-8B Pseudo-Labels

Full experiment run: **2 tasks × 3 modalities × 4 budgets × 3 seeds = 72 experiments**

| Modality | Model |
| --- | --- |
| text_only | BERTweet (`vinai/bertweet-base`) |
| image_only | CLIP ViT (`openai/clip-vit-base-patch32`) |
| text_image | BERTweet + CLIP ViT (late fusion) |

Results saved under `results/cotrain/lg-cotrain/qwen3-vl-8b/{run_id}/...`  
Completed experiments are automatically skipped on re-run.

**Prerequisite:** Run `03_zeroshot_qwen3-8B.ipynb` first, then generate pseudo-labels:
```bash
python scripts/create_pseudo_labels.py --model qwen3-vl-8b
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import time
import pandas as pd
from lg_cotrain.run_all import run_all_experiments, format_summary_table

In [2]:
# ---- Configuration ----
RUN_ID = "run-1"  # Change to run-2, run-3 etc. for different settings
NUM_GPUS = 2      # Parallel experiments across GPUs (1 = sequential)

PARAMS = dict(
    pseudo_label_source="qwen3-vl-8b",
    run_id=RUN_ID,
    # Paper defaults
    weight_gen_epochs=7,
    cotrain_epochs=10,
    finetune_max_epochs=100,
    finetune_patience=5,
    batch_size=32,
    lr=2e-5,
)

DATA_ROOT = os.path.abspath("../data")
RESULTS_ROOT = os.path.abspath("../results")

TASKS = ["informative", "humanitarian"]
MODALITIES = ["text_only", "image_only", "text_image"]
BUDGETS = [5, 10, 25, 50]
SEED_SETS = [1, 2, 3]

total = len(TASKS) * len(MODALITIES) * len(BUDGETS) * len(SEED_SETS)
print(f"Run ID:       {RUN_ID}")
print(f"GPUs:         {NUM_GPUS}")
print(f"Data root:    {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output path:  results/cotrain/lg-cotrain/qwen3-vl-8b/{RUN_ID}/{{task}}/{{modality}}/...")
print(f"Total experiments: {total}")

Run ID:       run-1
GPUs:         2
Data root:    D:\Workspace\Cotrain_CrisisMMD\data
Results root: D:\Workspace\Cotrain_CrisisMMD\results
Output path:  results/cotrain/lg-cotrain/qwen3-vl-8b/run-1/{task}/{modality}/...
Total experiments: 72


In [3]:
import gc, torch

all_summaries = {}  # (task, modality) -> results list

def run_batch(task, modality):
    """Run all 12 experiments for one (task, modality) and print summary."""
    print(f"\n{'='*70}")
    print(f"  {task} / {modality}  \u2014  {len(BUDGETS)}x{len(SEED_SETS)} = {len(BUDGETS)*len(SEED_SETS)} experiments")
    print(f"{'='*70}")
    
    n_total = len(BUDGETS) * len(SEED_SETS)
    progress = {"done": 0, "start": time.time()}

    def on_done(task, modality, budget, seed_set, status):
        progress["done"] += 1
        elapsed = time.time() - progress["start"]
        n = progress["done"]
        avg = elapsed / n
        eta = avg * (n_total - n)
        print(f"  [{n}/{n_total}] budget={budget}, seed={seed_set} -- {status} "
              f"({elapsed/60:.1f}min elapsed, ~{eta/60:.1f}min remaining)")

    start = time.time()
    results = run_all_experiments(
        task, modality,
        budgets=BUDGETS,
        seed_sets=SEED_SETS,
        num_gpus=NUM_GPUS,
        data_root=DATA_ROOT,
        results_root=RESULTS_ROOT,
        _on_experiment_done=on_done,
        **PARAMS,
    )
    elapsed = time.time() - start
    
    all_summaries[(task, modality)] = results
    
    print(f"\nCompleted {task}/{modality} in {elapsed/60:.1f}min")
    print(format_summary_table(results, task, modality,
                               budgets=BUDGETS, seed_sets=SEED_SETS))
    
    # Free GPU memory before next batch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

## Informative Task

In [4]:
run_batch("informative", "text_only")


  informative / text_only  —  4x3 = 12 experiments
Running 12 experiments in parallel across 2 GPUs...
  [1/12] budget=5, seed=2 -- done (22.7min elapsed, ~249.9min remaining)
  [2/12] budget=5, seed=1 -- done (25.5min elapsed, ~127.5min remaining)
  [3/12] budget=5, seed=3 -- done (45.2min elapsed, ~135.5min remaining)
  [4/12] budget=10, seed=1 -- done (51.3min elapsed, ~102.6min remaining)
  [5/12] budget=10, seed=2 -- done (68.2min elapsed, ~95.5min remaining)
  [6/12] budget=10, seed=3 -- done (78.1min elapsed, ~78.1min remaining)
  [7/12] budget=25, seed=1 -- done (90.5min elapsed, ~64.6min remaining)
  [8/12] budget=25, seed=2 -- done (103.6min elapsed, ~51.8min remaining)
  [9/12] budget=25, seed=3 -- done (112.8min elapsed, ~37.6min remaining)
  [10/12] budget=50, seed=1 -- done (127.9min elapsed, ~25.6min remaining)
  [11/12] budget=50, seed=2 -- done (134.1min elapsed, ~12.2min remaining)
  [12/12] budget=50, seed=3 -- done (152.6min elapsed, ~0.0min remaining)

Batch compl

[{'task': 'informative',
  'modality': 'text_only',
  'budget': 5,
  'seed_set': 1,
  'test_error_rate': np.float64(18.05736636245111),
  'test_macro_f1': 0.7562418432820195,
  'test_ece': 0.15326294559387996,
  'test_per_class_f1': [0.8803455723542116, 0.6321381142098274],
  'dev_error_rate': np.float64(19.453273998728548),
  'dev_macro_f1': 0.7326327787553991,
  'dev_ece': 0.16305587099224458,
  'stopping_strategy': 'baseline',
  'phase1_seed_strategy': 'last',
  'phase1_best_epoch': None,
  'lambda1_mean': 1.0763126179582012,
  'lambda1_std': 0.09757242636321625,
  'lambda2_mean': 0.7860059916217048,
  'lambda2_std': 0.09975472254883398},
 {'task': 'informative',
  'modality': 'text_only',
  'budget': 5,
  'seed_set': 2,
  'test_error_rate': np.float64(17.07953063885267),
  'test_macro_f1': 0.7724705615942029,
  'test_ece': 0.11095571401377234,
  'test_per_class_f1': [0.8860869565217391, 0.6588541666666666],
  'dev_error_rate': np.float64(18.817546090273364),
  'dev_macro_f1': 0.746

In [ ]:
run_batch("informative", "image_only")


  informative / image_only  —  4x3 = 12 experiments
budget=5, seed=1 -- SKIPPED (exists)
  [1/12] budget=5, seed=1 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=5, seed=2 -- SKIPPED (exists)
  [2/12] budget=5, seed=2 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=5, seed=3 -- SKIPPED (exists)
  [3/12] budget=5, seed=3 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=10, seed=1 -- SKIPPED (exists)
  [4/12] budget=10, seed=1 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=10, seed=2 -- SKIPPED (exists)
  [5/12] budget=10, seed=2 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=10, seed=3 -- SKIPPED (exists)
  [6/12] budget=10, seed=3 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=25, seed=1 -- SKIPPED (exists)
  [7/12] budget=25, seed=1 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=25, seed=2 -- SKIPPED (exists)
  [8/12] budget=25, seed=2 -- skipped (0.0min elapsed, ~0.0min remaining)
budget=25, seed=3 -- SKIPPED (exists)
  [9/12] budget=25,

In [ ]:
run_batch("informative", "text_image")

## Humanitarian Task

In [ ]:
run_batch("humanitarian", "text_only")

In [ ]:
run_batch("humanitarian", "image_only")

In [ ]:
run_batch("humanitarian", "text_image")

## Cross-Modality Summary

In [ ]:
import numpy as np

rows = []
for (task, modality), results in all_summaries.items():
    valid = [r for r in results if r is not None]
    if not valid:
        rows.append({"Task": task, "Modality": modality,
                     "N": 0, "Mean F1": "-", "Std F1": "-",
                     "Mean Err%": "-", "Mean ECE": "-"})
        continue
    f1s = [r["test_macro_f1"] for r in valid]
    errs = [r["test_error_rate"] for r in valid]
    eces = [r["test_ece"] for r in valid]
    rows.append({
        "Task": task,
        "Modality": modality,
        "N": len(valid),
        "Mean F1": f"{np.mean(f1s):.4f}",
        "Std F1": f"{np.std(f1s):.4f}",
        "Mean Err%": f"{np.mean(errs):.2f}",
        "Mean ECE": f"{np.mean(eces):.4f}",
    })

df = pd.DataFrame(rows)
print(f"Cross-Modality Summary ({RUN_ID})")
print(f"Averaged over {len(BUDGETS)} budgets x {len(SEED_SETS)} seeds = {len(BUDGETS)*len(SEED_SETS)} experiments each")
print("=" * 70)
df

## Cleanup

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory freed. Peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("No GPU available (ran on CPU)")

n_success = sum(1 for results in all_summaries.values() for r in results if r is not None)
print(f"\nDone. {n_success}/{total} experiments completed successfully.")